***P3***


In [15]:
from pathlib import Path
import requests
import zipfile
import io

# GitHub dataset URL
url = "https://raw.githubusercontent.com/asmaramraj/dsa405-project/main/data/raw/archive.zip"

# Create the data folder
folder = Path("data/raw")
folder.mkdir(parents=True, exist_ok=True)

zip_path = folder / "archive.zip"

# Download only if not already cached
if zip_path.exists() and zip_path.stat().st_size > 0:
    print("Using cached ZIP file.")
else:
    print("Downloading ZIP from GitHub...")

    response = requests.get(
        url,
        headers={"User-Agent": "DSA405-Student-Project/1.0 (coursework)"},
        timeout=30
    )

    # Stop if the request fails
    response.raise_for_status()

    # Make sure the response is a ZIP file
    if not zipfile.is_zipfile(io.BytesIO(response.content)):
        raise ValueError("Downloaded file is not a valid ZIP archive!")

    # Save the ZIP
    zip_path.write_bytes(response.content)
    print("Downloaded and saved ZIP successfully!")

# Verify the file exists
assert zip_path.exists(), "ZIP file was not saved!"
assert zipfile.is_zipfile(zip_path), "Saved file is not a valid ZIP!"

print("PASS: ZIP file saved successfully!")
print("File size:", zip_path.stat().st_size, "bytes")
print("Saved at:", zip_path.resolve())

Downloaded and saved ZIP successfully!
PASS: ZIP file saved successfully!
File size: 22428121 bytes
Saved at: /content/data/raw/archive.zip


In [21]:
import time
import zipfile
from pathlib import Path
import requests
import pandas as pd


HEADERS = {"User-Agent": "NCState-DSA405-Student (avni.maramraj@ncsu.edu)"}
hornets = Path("data/raw")
hornets.mkdir(parents=True, exist_ok=True)
zip_path = hornets / "archive.zip"
csv_path = hornets / "scoring.csv"


def load_hornets_data(url):
    """Downloads archive.zip, extracts scoring.csv, caches it, and returns the DataFrame."""
    # If the CSV already exists, load directly from cache
    if csv_path.is_file():
        print("Loading scoring.csv from local cache...")
        return pd.read_csv(csv_path, engine= 'python', on_bad_lines='skip')

    print("Downloading archive.zip from GitHub...")
    time.sleep(1) # 1-second rate limit

    response = requests.get(url, headers=HEADERS, timeout=60)

    if response.status_code != 200:
        raise RuntimeError(f"Failed loudly: HTTP {response.status_code}")

    # Write binary zip content
    zip_path.write_bytes(response.content)

    print("Extracting scoring.csv from archive...")
    with zipfile.ZipFile(zip_path, 'r') as archive:
        file_list = archive.namelist()
        print(f"Files inside archive.zip: {file_list}")

        target_file = next(
    (f for f in file_list if Path(f).name == "scoring.csv"),
    None
)

        if target_file is None:
          raise FileNotFoundError(
        "Failed loudly: scoring.csv was not found in the ZIP!"
    )

        with archive.open(target_file) as source, open(csv_path, "wb") as destination:
            destination.write(source.read())

    print("Successfully extracted and cached scoring.csv!")
    return pd.read_csv(csv_path)

# 2. Run the acquisition function
URL = "https://raw.githubusercontent.com/asmaramraj/dsa405-project/main/data/raw/archive.zip"

df = load_hornets_data(URL)
print(df.head())


Loading scoring.csv from local cache...
     gameid        date     type  playerid          player team home away  \
0  29600001  1996-11-01  regular       103        Todd Day  BOS  BOS  CHI   
1  29600001  1996-11-01  regular       677   Eric Williams  BOS  BOS  CHI   
2  29600001  1996-11-01  regular       952  Antoine Walker  BOS  BOS  CHI   
3  29600001  1996-11-01  regular       133    David Wesley  BOS  BOS  CHI   
4  29600001  1996-11-01  regular      1136     Brett Szabo  BOS  BOS  CHI   

   MIN  %FGA 2PT  ...  %PTS OFFTO  %PTS PITP  2FGM %AST  2FGM %UAST  \
0   20      71.4  ...         0.0       44.4       50.0        50.0   
1   25     100.0  ...        14.3       85.7       66.7        33.3   
2   27      75.0  ...         0.0       50.0       50.0        50.0   
3   41      75.0  ...        26.3       63.2       42.9        57.1   
4    2       0.0  ...         0.0        0.0        0.0         0.0   

   3FGM %AST  3FGM %UAST  FGM %AST  FGM %UAST  win  season  
0      10

In [23]:
# Demonstrate that an invalid website causes an error
bad_url = "https://example.com/this-file-does-not-exist.zip"

try:
    response = requests.get(
        bad_url,
        timeout=15,
        headers={"User-Agent": "DSA405-course-project/1.0"}
    )
    response.raise_for_status()

except requests.exceptions.HTTPError as error:
    print("PASS: Invalid URL was caught!")
    print("Error:", error)

PASS: Invalid URL was caught!
Error: 404 Client Error: Not Found for url: https://example.com/this-file-does-not-exist.zip


In [22]:
# Extract the second CSV from the cached ZIP
team_path = hornets / "team_scoring.csv"

with zipfile.ZipFile(zip_path, "r") as archive:
    matches = [
        name for name in archive.namelist()
        if Path(name).name == "team_scoring.csv"
    ]

    if len(matches) != 1:
        raise FileNotFoundError(
            "Expected exactly one team_scoring.csv in ZIP!"
        )

    if not team_path.exists():
        team_path.write_bytes(archive.read(matches[0]))

# Load both CSVs
scoring = pd.read_csv(csv_path)
team_scoring = pd.read_csv(team_path)

print("Scoring rows:", len(scoring))
print("Team scoring rows:", len(team_scoring))
print("Both datasets loaded successfully!")

Scoring rows: 761275
Team scoring rows: 76046
Both datasets loaded successfully!


### Join Design

**Join key:** `gameid`

**Left table:** Charlotte Hornets game records from `team_scoring.csv`, with one row per Hornets game.

**Right table:** Opposing team scoring records from `team_scoring.csv`, with one row per opponent per Hornets game.

**Relationship:** One-to-one because each Hornets game should have one matching opponent record.

**Expected row count:** 82 games.

**Join type:** Inner join because only games with both Charlotte and opponent records are needed for the analysis.

**Unmatched rows:** An inner join would remove unmatched games. I will check for unmatched game IDs before merging to make sure no games are lost.

**Potential errors:** Using `validate="one_to_one"` prevents fan-out caused by duplicate game IDs. Checking unmatched keys before merging prevents a join from silently removing games.

In [24]:
# Filter 2025-2026 regular season
season = team_scoring[
    (team_scoring["season"] == 2026) &
    (team_scoring["type"] == "regular")
].copy()

# Charlotte Hornets games
cha = season[season["team"] == "CHA"].copy()

# Opponent scoring data for Hornets games
opp = season[
    (season["team"] != "CHA") &
    (season["gameid"].isin(cha["gameid"]))
][["gameid", "team", "%PTS PITP"]].copy()

opp = opp.rename(columns={
    "team": "opponent",
    "%PTS PITP": "opponent_paint_pct"
})

# Check unmatched games before merging
unmatched = set(cha["gameid"]) - set(opp["gameid"])
print("Unmatched games:", len(unmatched))

# Merge the two tables
games = cha.merge(
    opp,
    on="gameid",
    how="inner",
    validate="one_to_one",
    indicator=True
)

print("Joined rows:", len(games))
print(games[["gameid", "team", "opponent", "opponent_paint_pct"]].head())

Unmatched games: 0
Joined rows: 82
     gameid team opponent  opponent_paint_pct
0  22500080  CHA      BKN                39.3
1  22500102  CHA      PHI                32.0
2  22500109  CHA      WAS                40.7
3  22500124  CHA      MIA                40.3
4  22500138  CHA      ORL                43.9


In [25]:
# 1. Protects against a join adding or losing games
assert len(games) == 82, "Incorrect number of games!"

# 2. Protects against duplicate game records
assert games["gameid"].is_unique, "Duplicate game IDs!"

# 3. Protects against missing join keys
assert games["gameid"].notna().all(), "Missing game IDs!"

# 4. Protects against unmatched games
assert (games["_merge"] == "both").all(), "Unmatched records!"

# 5. Protects against invalid percentages
assert games["opponent_paint_pct"].between(0, 100).all(), \
    "Invalid paint scoring percentage!"

# 6. Protects against an incorrect home/away split
assert (games["home"] == "CHA").sum() == 41, \
    "Incorrect number of home games!"

assert (games["home"] != "CHA").sum() == 41, \
    "Incorrect number of away games!"

print("PASS: All 7 verification checks passed!")
print("Verified rows:", len(games))
print("Unique games:", games["gameid"].nunique())
print("Missing join keys:", games["gameid"].isna().sum())

PASS: All 7 verification checks passed!
Verified rows: 82
Unique games: 82
Missing join keys: 0


### Ethics and Legality

For this project, I used publicly available NBA scoring data that I originally downloaded from Kaggle. For my acquisition code, I retrieved the dataset from my GitHub repository using an HTTP request. This is different from directly downloading the data from Kaggle, so I wrote down the change in my acquisition method. My code doesn't require a login, bypass a paywall, or collect personal information. I also used a one-second delay, a User-Agent identifying my coursework, and local caching to avoid unnecessary requests.

### One Thing That Surprised Me

One thing that surprised me was that all 82 Charlotte Hornets games matched correctly with their opponents' records. After completing the join, I expected there might be missing or duplicate games, but my verification checks confirmed that no games were lost or duplicated. I also learned how important it is to verify a join rather than assuming it worked just because Python didn't return an error.